# DBSCAN -- UCI Seeds Dataset

This notebook demonstrates density-based spatial clustering (DBSCAN) applied to wheat kernel
physical measurements, where the goal is to recover three known seed varieties as clusters
without specifying the number of clusters in advance.

- **Core/border/noise classification:** DBSCAN assigns each point to one of three roles based on
  neighbourhood density
- **Hyperparameter sensitivity:** the effect of `eps` and `min_samples` on cluster structure
- **Ground-truth comparison:** detected clusters are compared against true variety labels using
  Adjusted Rand Score
- **DBSCAN vs KMeans:** both methods are applied to the same data to contrast density-based
  and centroid-based partitioning
- All clustering is performed with `mlpackage.DBSCAN`; no sklearn clustering is used

## Mathematical Intuition

### Core, Border, and Noise Points

Given radius $\varepsilon > 0$ and minimum neighbourhood size $m$:

- **Core point:** $|\mathcal{N}_\varepsilon(p)| \geq m$ -- has at least $m$ neighbours within distance $\varepsilon$
- **Border point:** $|\mathcal{N}_\varepsilon(p)| < m$ but $p$ lies within $\varepsilon$ of some core point
- **Noise point:** neither core nor border; assigned label $-1$

### Density-Reachability

Point $q$ is **directly density-reachable** from core point $p$ if:

$$\text{dist}(p, q) \leq \varepsilon$$

A cluster is the maximal set of mutually density-connected points. Because clusters are defined
by connected dense regions, DBSCAN recovers arbitrary shapes -- rings, crescents, interleaved
spirals -- that KMeans cannot separate (KMeans assumes convex, roughly spherical clusters).

### Why DBSCAN Outperforms KMeans on Non-Convex Data

KMeans assigns each point to the nearest centroid, which partitions space into Voronoi regions
(always convex). DBSCAN grows clusters from seed points outward through density chains, so the
cluster boundary can follow any shape that maintains local density $\geq m$ within radius
$\varepsilon$.

## Dataset Overview

**Source:** UCI Seeds Dataset (ID 236) -- geometric measurements of wheat kernels from three
varieties | **Rows:** 210 | **Features used:** Area, Perimeter (2D for direct visualisation)

| Feature | Type | Description |
|---|---|---|
| Area | Continuous | Area of kernel cross-section (A) |
| Perimeter | Continuous | Perimeter of kernel (P) |
| Label | Categorical | True variety: 1=Kama, 2=Rosa, 3=Canadian |

The three varieties are separable in (Area, Perimeter) space, making this dataset well-suited
for demonstrating DBSCAN's ability to find compact clusters and flag boundary noise without
knowing the number of clusters in advance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

from ucimlrepo import fetch_ucirepo
from sklearn.metrics import adjusted_rand_score
from mlpackage import DBSCAN, KMeans, StandardScaler

seeds      = fetch_ucirepo(id=236)
X_raw      = seeds.data.features.values.astype(float)
y_true     = seeds.data.targets.values.ravel().astype(int)   # 1=Kama, 2=Rosa, 3=Canadian
feat_names = list(seeds.data.features.columns)

# Use Area and Perimeter (first two features) for 2D visualisation
X_2d       = X_raw[:, :2]

print(f"Samples  : {X_raw.shape[0]}  |  Total features: {X_raw.shape[1]}")
print(f"Features : {feat_names[:2]}")
print(f"Kama={np.sum(y_true==1)}, Rosa={np.sum(y_true==2)}, Canadian={np.sum(y_true==3)}")

## Exploratory Data Analysis

In [ ]:
variety_names = {1: "Kama", 2: "Rosa", 3: "Canadian"}
colors        = {1: "steelblue", 2: "coral", 3: "seagreen"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

counts = [np.sum(y_true == k) for k in [1, 2, 3]]
axes[0].bar([variety_names[k] for k in [1, 2, 3]], counts,
            color=[colors[k] for k in [1, 2, 3]])
axes[0].set_title("Seed Variety Counts")
axes[0].set_xlabel("Variety")
axes[0].set_ylabel("Count")

for k in [1, 2, 3]:
    m = y_true == k
    axes[1].scatter(X_2d[m, 0], X_2d[m, 1], color=colors[k],
                    label=variety_names[k], alpha=0.7, s=30)
axes[1].set_title("Area vs Perimeter by True Variety")
axes[1].set_xlabel(feat_names[0])
axes[1].set_ylabel(feat_names[1])
axes[1].legend()

plt.tight_layout()
plt.show()

## Preprocessing

In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_2d)

print(f"Samples: {X_scaled.shape[0]}  |  Features (standardised): {feat_names[0]}, {feat_names[1]}")

## Baseline DBSCAN (eps=0.5, min_samples=5)

In [ ]:
db     = DBSCAN(eps=0.5, min_samples=5)
db.fit(X_scaled)
labels = db.labels_

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = int(np.sum(labels == -1))
ars        = adjusted_rand_score(y_true, labels)
print(f"Clusters found: {n_clusters}  |  Noise points: {n_noise}  |  ARS: {ars:.4f}")

cmap          = plt.get_cmap("tab10")
unique_labels = sorted(set(labels))

plt.figure(figsize=(7, 5))
for lbl in unique_labels:
    m = labels == lbl
    if lbl == -1:
        plt.scatter(X_scaled[m, 0], X_scaled[m, 1],
                    color="black", marker="x", s=50, alpha=0.7, label="Noise")
    else:
        plt.scatter(X_scaled[m, 0], X_scaled[m, 1],
                    color=cmap(lbl % 10), s=30, alpha=0.7, label=f"Cluster {lbl}")
plt.title(f"DBSCAN (eps=0.5, min_samples=5) -- {n_clusters} clusters, {n_noise} noise  |  ARS={ars:.3f}")
plt.xlabel(f"{feat_names[0]} (standardised)")
plt.ylabel(f"{feat_names[1]} (standardised)")
plt.legend()
plt.tight_layout()
plt.show()

## Hyperparameter Sensitivity

In [ ]:
eps_values = [0.2, 0.5, 0.8, 1.5]
fig, axes  = plt.subplots(1, 4, figsize=(18, 4))

for ax, eps_val in zip(axes, eps_values):
    db_e = DBSCAN(eps=eps_val, min_samples=5)
    db_e.fit(X_scaled)
    lbl  = db_e.labels_
    nc   = len(set(lbl)) - (1 if -1 in lbl else 0)
    nn   = int(np.sum(lbl == -1))
    for l in sorted(set(lbl)):
        m = lbl == l
        if l == -1:
            ax.scatter(X_scaled[m, 0], X_scaled[m, 1], color="black", marker="x", s=20, alpha=0.5)
        else:
            ax.scatter(X_scaled[m, 0], X_scaled[m, 1], color=cmap(l % 10), s=20, alpha=0.6)
    ax.set_title(f"eps={eps_val}\n{nc} clusters, {nn} noise", fontsize=9)
    ax.set_xlabel(feat_names[0])
    ax.set_ylabel(feat_names[1])
plt.suptitle("DBSCAN: eps Sensitivity (min_samples=5)")
plt.tight_layout()
plt.show()

In [ ]:
ms_values = [3, 5, 10, 20]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, ms in zip(axes, ms_values):
    db_m = DBSCAN(eps=0.5, min_samples=ms)
    db_m.fit(X_scaled)
    lbl  = db_m.labels_
    nc   = len(set(lbl)) - (1 if -1 in lbl else 0)
    nn   = int(np.sum(lbl == -1))
    for l in sorted(set(lbl)):
        m = lbl == l
        if l == -1:
            ax.scatter(X_scaled[m, 0], X_scaled[m, 1], color="black", marker="x", s=20, alpha=0.5)
        else:
            ax.scatter(X_scaled[m, 0], X_scaled[m, 1], color=cmap(l % 10), s=20, alpha=0.6)
    ax.set_title(f"min_samples={ms}\n{nc} clusters, {nn} noise", fontsize=9)
    ax.set_xlabel(feat_names[0])
    ax.set_ylabel(feat_names[1])
plt.suptitle("DBSCAN: min_samples Sensitivity (eps=0.5)")
plt.tight_layout()
plt.show()

In [ ]:
km        = KMeans(n_clusters=3, random_state=42)
km.fit(X_scaled)
km_labels = km.labels_
db_labels = db.labels_

km_ars = adjusted_rand_score(y_true, km_labels)
print(f"DBSCAN ARS: {ars:.4f}  |  KMeans ARS: {km_ars:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for l in sorted(set(db_labels)):
    m = db_labels == l
    if l == -1:
        axes[0].scatter(X_scaled[m, 0], X_scaled[m, 1], color="black", marker="x", s=40, alpha=0.6, label="Noise")
    else:
        axes[0].scatter(X_scaled[m, 0], X_scaled[m, 1], color=cmap(l % 10), s=25, alpha=0.7, label=f"C{l}")
axes[0].set_title(f"DBSCAN (eps=0.5, min_samples=5)  |  ARS={ars:.3f}")
axes[0].set_xlabel(f"{feat_names[0]} (standardised)")
axes[0].set_ylabel(f"{feat_names[1]} (standardised)")
axes[0].legend()

for l in range(3):
    m = km_labels == l
    axes[1].scatter(X_scaled[m, 0], X_scaled[m, 1], color=cmap(l), s=25, alpha=0.7, label=f"C{l}")
axes[1].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                color="black", marker="X", s=150, zorder=6, label="Centroid")
axes[1].set_title(f"KMeans (k=3)  |  ARS={km_ars:.3f}")
axes[1].set_xlabel(f"{feat_names[0]} (standardised)")
axes[1].set_ylabel(f"{feat_names[1]} (standardised)")
axes[1].legend()

plt.suptitle("DBSCAN vs KMeans on Seeds Data")
plt.tight_layout()
plt.show()

## Interpretation and Conclusions

_Analysis to be completed after running the notebook end-to-end._